In [17]:
import pandas as pd
import numpy as np
import pyreadstat
from pathlib import Path

# --- Path resolution (identical convention to Birth_History_Recon.ipynb) ---
_cwd = Path.cwd()
if (_cwd / "data" / "raw" / "DHS_Districts").is_dir():
    REPO_ROOT = _cwd
elif (_cwd.parent / "data" / "raw" / "DHS_Districts").is_dir():
    REPO_ROOT = _cwd.parent
else:
    raise FileNotFoundError(f"Could not find data/raw/DHS_Districts from cwd {_cwd}.")

MICRO_DIR = REPO_ROOT / "data" / "raw" / "DHS_microdata"
CROSSWALK_DIR = REPO_ROOT / "data" / "processed" / "district_crosswalks"
OUT_DIR = REPO_ROOT / "data" / "processed"

IR4_PATH = MICRO_DIR / "NFHS4_2015-16_IndividualRecode" / "IAIR74FL.DTA"
IR5_PATH = MICRO_DIR / "NFHS5_2019-21_IndividualRecode" / "IAIR7EFL.DTA"

for p in [IR4_PATH, IR5_PATH]:
    print(p, "->", p.exists())

# --- Confirm column availability before reading full files ---
def peek(path, row_limit=5):
    df, meta = pyreadstat.read_dta(str(path), row_limit=row_limit)
    return df, meta

_, ir4_meta = peek(IR4_PATH)
_, ir5_meta = peek(IR5_PATH)
ir4_cols, ir5_cols = set(ir4_meta.column_names), set(ir5_meta.column_names)

# --- Mobile phone / internet variables ---
# v169a: owns a mobile phone (NFHS-4 and NFHS-5)
# v169c: owns a smartphone (NFHS-5 only)
# v171a: used internet in past 12 months (NFHS-5; limited/absent in NFHS-4)
# v171b: frequency of internet use (NFHS-5 only)
CANDIDATE_VARS = ["v169a", "v169c", "v171a", "v171b"]

ir4_present = [v for v in CANDIDATE_VARS if v in ir4_cols]
ir5_present = [v for v in CANDIDATE_VARS if v in ir5_cols]
print("Present in IR4:", ir4_present)
print("Present in IR5:", ir5_present)

missing_ir4 = [v for v in CANDIDATE_VARS if v not in ir4_cols]
missing_ir5 = [v for v in CANDIDATE_VARS if v not in ir5_cols]
if missing_ir4:
    print(f"NOTE: {missing_ir4} not present in IR4 (expected for v169c/v171b, NFHS-5-only variables)")
if missing_ir5:
    print(f"WARNING: {missing_ir5} unexpectedly missing from IR5 -- check DHS-8 recode manual")

IR4_VARS = ["caseid", "sdistri", "v005"] + ir4_present
IR5_VARS = ["caseid", "sdist", "v005"] + ir5_present

ir4, _ = pyreadstat.read_dta(str(IR4_PATH), usecols=IR4_VARS, apply_value_formats=False)
ir5, _ = pyreadstat.read_dta(str(IR5_PATH), usecols=IR5_VARS, apply_value_formats=False)

ir4 = ir4.rename(columns={"sdistri": "district_code_raw"})
ir5 = ir5.rename(columns={"sdist": "district_code_raw"})
print("IR4 shape:", ir4.shape, "| IR5 shape:", ir5.shape)

# --- Crosswalk mapping (identical logic to Birth_History_Recon.ipynb) ---
crosswalk = pd.read_csv(CROSSWALK_DIR / "nfhs4_nfhs5_stable_district_crosswalk.csv")
nfhs4_to_stable = dict(zip(crosswalk["district_code"], crosswalk["district_code"]))
nfhs5_to_stable = dict(zip(crosswalk["district_code_nfhs5"], crosswalk["district_code"]))

ir4["stable_district_id"] = ir4["district_code_raw"].map(nfhs4_to_stable)
ir5["stable_district_id"] = ir5["district_code_raw"].map(nfhs5_to_stable)

ir4_unmapped = ir4["stable_district_id"].isna().sum()
ir5_unmapped = ir5["stable_district_id"].isna().sum()
print(f"IR4: {ir4_unmapped} rows unmapped ({ir4_unmapped/len(ir4):.1%}) -- expect ~9.4%, matching Birth_History_Recon")
print(f"IR5: {ir5_unmapped} rows unmapped ({ir5_unmapped/len(ir5):.1%}) -- expect ~18.5%, matching Birth_History_Recon")

ir4_full = ir4[ir4["stable_district_id"].notna()].copy()
ir5_full = ir5[ir5["stable_district_id"].notna()].copy()
ir4_full["weight"] = ir4_full["v005"] / 1_000_000
ir5_full["weight"] = ir5_full["v005"] / 1_000_000

print("Final scoped IR4:", ir4_full.shape, "| Final scoped IR5:", ir5_full.shape)
ir4_full.head()

/Users/eknoorsandhu/DHS_India_Research/data/raw/DHS_microdata/NFHS4_2015-16_IndividualRecode/IAIR74FL.DTA -> True
/Users/eknoorsandhu/DHS_India_Research/data/raw/DHS_microdata/NFHS5_2019-21_IndividualRecode/IAIR7EFL.DTA -> True
Present in IR4: []
Present in IR5: ['v169a', 'v171a', 'v171b']
NOTE: ['v169a', 'v169c', 'v171a', 'v171b'] not present in IR4 (expected for v169c/v171b, NFHS-5-only variables)
IR4 shape: (699686, 3) | IR5 shape: (724115, 6)
IR4: 65884 rows unmapped (9.4%) -- expect ~9.4%, matching Birth_History_Recon
IR5: 133650 rows unmapped (18.5%) -- expect ~18.5%, matching Birth_History_Recon
Final scoped IR4: (633802, 5) | Final scoped IR5: (590465, 8)


,caseid,v005,district_code_raw,stable_district_id,weight
0,01000101 02,191760,640,640.0,0.19176
1,01000101 04,191760,640,640.0,0.19176
2,01000109 01,191760,640,640.0,0.19176
3,01000109 02,191760,640,640.0,0.19176
4,01000109 03,191760,640,640.0,0.19176


In [18]:
# --- Diagnostic: search full column labels for mobile/phone/internet/smartphone content ---

def find_candidates(meta, keywords):
    hits = []
    for name, label in zip(meta.column_names, meta.column_labels or []):
        label_str = (label or "").lower()
        if any(kw in label_str for kw in keywords) or any(kw in name.lower() for kw in keywords):
            hits.append((name, label))
    return hits

keywords = ["mobile", "phone", "smart", "internet", "cell"]

print("=== IR4 (NFHS-4) candidates ===")
for name, label in find_candidates(ir4_meta, keywords):
    print(f"  {name}: {label}")

print("\n=== IR5 (NFHS-5) candidates ===")
for name, label in find_candidates(ir5_meta, keywords):
    print(f"  {name}: {label}")

# --- Diagnostic: is v171b genuinely empty, or a read issue? ---
# Re-read v171b directly from the raw file (unscoped, no usecols restriction
# beyond itself) to rule out anything introduced by our own pipeline
v171b_raw, _ = pyreadstat.read_dta(str(IR5_PATH), usecols=["caseid", "v171a", "v171b"], apply_value_formats=False)
print(f"\nv171b raw non-null count: {v171b_raw['v171b'].notna().sum()} / {len(v171b_raw)}")
print(f"v171b dtype: {v171b_raw['v171b'].dtype}")
print(f"v171b value_counts (including any non-standard missing codes):")
print(v171b_raw["v171b"].value_counts(dropna=False).head(15))

=== IR4 (NFHS-4) candidates ===
  v153: household has: telephone (land-line)
  v3a00h: source of fp for non-users: govt. mobile clinic
  v3a00m: source of fp for non-users: mobile clinic
  h12h_1: diarrhea: public: govt. mobile clinic
  h12h_2: diarrhea: public: govt. mobile clinic
  h12h_3: diarrhea: public: govt. mobile clinic
  h12h_4: diarrhea: public: govt. mobile clinic
  h12h_5: diarrhea: public: govt. mobile clinic
  h12h_6: diarrhea: public: govt. mobile clinic
  h32h_1: fever/cough: public: govt. mobile clinic
  h32h_2: fever/cough: public: govt. mobile clinic
  h32h_3: fever/cough: public: govt. mobile clinic
  h32h_4: fever/cough: public: govt. mobile clinic
  h32h_5: fever/cough: public: govt. mobile clinic
  h32h_6: fever/cough: public: govt. mobile clinic
  v762ah: source for condoms: public: govt. mobile clinic
  v762am: source for condoms: private: mobile clinic
  v770f: sought sti advice/treatment from: public: mobile clinic
  v770o: sought sti advice/treatment from: 

In [19]:
# --- Revised Segment 2: load state-module variables with the CORRECT module-specific weights ---

IR4_MODULE_VARS = ["caseid", "s930", "sv005"]
IR5_MODULE_VARS = ["caseid", "v169a", "s934", "sweight"]

ir4_mod, _ = pyreadstat.read_dta(str(IR4_PATH), usecols=IR4_MODULE_VARS, apply_value_formats=False)
ir5_mod, _ = pyreadstat.read_dta(str(IR5_PATH), usecols=IR5_MODULE_VARS, apply_value_formats=False)

ir4_mod["state_weight"] = ir4_mod["sv005"] / 1_000_000
ir5_mod["state_weight"] = ir5_mod["sweight"] / 1_000_000

# Merge onto the already-scoped, district-mapped samples (reuse ir4_full/ir5_full's district mapping)
ir4_mod = ir4_mod.merge(ir4_full[["caseid", "stable_district_id"]], on="caseid", how="inner")
ir5_mod = ir5_mod.merge(ir5_full[["caseid", "stable_district_id"]], on="caseid", how="inner")

def weighted_mean(df, value_col, weight_col):
    valid = df[value_col].notna()
    if valid.sum() == 0:
        return np.nan
    return np.average(df.loc[valid, value_col], weights=df.loc[valid, weight_col])

def module_district_agg(df, value_col, weight_col, round_label):
    rows = []
    for district_id, g in df.groupby("stable_district_id"):
        n_valid = g[value_col].notna().sum()
        rows.append({
            "stable_district_id": district_id,
            "round": round_label,
            "n_module_respondents": n_valid,
            f"{value_col}_mean": weighted_mean(g, value_col, weight_col)
        })
    return pd.DataFrame(rows)

district_ir4_mobile = module_district_agg(ir4_mod, "s930", "state_weight", "NFHS-4")
district_ir5_mobile = module_district_agg(ir5_mod, "v169a", "state_weight", "NFHS-5")
district_ir5_internet = module_district_agg(ir5_mod, "s934", "state_weight", "NFHS-5")

MIN_MODULE_N = 30  # matches the project's established MIN_CELL_N convention
for name, d in [("NFHS-4 mobile (s930)", district_ir4_mobile),
                ("NFHS-5 mobile (v169a)", district_ir5_mobile),
                ("NFHS-5 internet (s934)", district_ir5_internet)]:
    low_n = (d["n_module_respondents"] < MIN_MODULE_N).sum()
    print(f"{name}: {low_n} / {len(d)} districts flagged low module-N (<{MIN_MODULE_N})")
    print(d.describe())
    print()

NFHS-4 mobile (s930): 0 / 575 districts flagged low module-N (<30)
       stable_district_id  n_module_respondents   s930_mean
count          575.000000            575.000000  575.000000
mean           320.266087            192.528696    0.454949
std            187.335890             65.200726    0.171515
min              1.000000             54.000000    0.107459
25%            161.500000            152.000000    0.329433
50%            324.000000            173.000000    0.432787
75%            483.500000            207.500000    0.564904
max            640.000000            483.000000    0.905161

NFHS-5 mobile (v169a): 1 / 575 districts flagged low module-N (<30)
       stable_district_id  n_module_respondents  v169a_mean
count          575.000000            575.000000  575.000000
mean           320.266087            154.206957    0.556034
std            187.335890             30.881034    0.175110
min              1.000000             14.000000    0.175709
25%            161.50000

In [20]:
# --- Segment 3: Merge, construct candidate intensity measures, check agreement ---

district_intensity = (
    district_ir4_mobile[["stable_district_id", "n_module_respondents", "s930_mean"]]
    .rename(columns={"n_module_respondents": "n_resp_nfhs4"})
    .merge(
        district_ir5_mobile[["stable_district_id", "n_module_respondents", "v169a_mean"]]
        .rename(columns={"n_module_respondents": "n_resp_nfhs5"}),
        on="stable_district_id", how="outer"
    )
    .merge(
        district_ir5_internet[["stable_district_id", "s934_mean"]],
        on="stable_district_id", how="outer"
    )
)

# --- Candidate measures ---
district_intensity["delta_mobile_ownership"] = district_intensity["v169a_mean"] - district_intensity["s930_mean"]
district_intensity["nfhs5_mobile_level"] = district_intensity["v169a_mean"]
district_intensity["nfhs5_internet_level"] = district_intensity["s934_mean"]

district_intensity["low_n_either_round"] = (
    (district_intensity["n_resp_nfhs4"] < MIN_MODULE_N) | (district_intensity["n_resp_nfhs5"] < MIN_MODULE_N)
)
print(f"Districts flagged low-N in either round: {district_intensity['low_n_either_round'].sum()} / {len(district_intensity)}")

# --- Intensity classification (terciles) for each candidate measure ---
for col in ["delta_mobile_ownership", "nfhs5_mobile_level", "nfhs5_internet_level"]:
    district_intensity[f"{col}_tercile"] = pd.qcut(
        district_intensity[col], q=3, labels=["low", "mid", "high"]
    )

print("\nDistrict-level intensity measure summary:")
print(district_intensity[["delta_mobile_ownership", "nfhs5_mobile_level", "nfhs5_internet_level"]].describe())

# --- Do the three measures agree on which districts are "high intensity"? ---
corr = district_intensity[["delta_mobile_ownership", "nfhs5_mobile_level", "nfhs5_internet_level"]].corr()
print("\nCorrelation between candidate intensity measures:")
print(corr)

print("\nTercile agreement: delta_mobile_ownership vs nfhs5_internet_level")
print(pd.crosstab(district_intensity["delta_mobile_ownership_tercile"],
                   district_intensity["nfhs5_internet_level_tercile"]))

print("\nTercile agreement: nfhs5_mobile_level vs nfhs5_internet_level")
print(pd.crosstab(district_intensity["nfhs5_mobile_level_tercile"],
                   district_intensity["nfhs5_internet_level_tercile"]))

district_intensity.sort_values("stable_district_id").head(10)

Districts flagged low-N in either round: 1 / 575

District-level intensity measure summary:
       delta_mobile_ownership  nfhs5_mobile_level  nfhs5_internet_level
count              575.000000          575.000000            575.000000
mean                 0.101085            0.556034              0.344914
std                  0.094991            0.175110              0.159820
min                 -0.234382            0.175709              0.069841
25%                  0.042234            0.413204              0.216860
50%                  0.099639            0.544568              0.311405
75%                  0.161827            0.689550              0.457069
max                  0.405859            0.933052              0.849537

Correlation between candidate intensity measures:
                        delta_mobile_ownership  nfhs5_mobile_level  \
delta_mobile_ownership                1.000000            0.308690   
nfhs5_mobile_level                    0.308690            1.000000   

,stable_district_id,n_resp_nfhs4,s930_mean,n_resp_nfhs5,v169a_mean,s934_mean,delta_mobile_ownership,nfhs5_mobile_level,nfhs5_internet_level,low_n_either_round,delta_mobile_ownership_tercile,nfhs5_mobile_level_tercile,nfhs5_internet_level_tercile
0,1.0,305,0.482354,182,0.709814,0.454159,0.227460,0.709814,0.454159,False,high,high,high
1,2.0,361,0.529261,189,0.679513,0.446047,0.150252,0.679513,0.446047,False,high,high,high
2,3.0,242,0.872892,177,0.821477,0.625899,-0.051416,0.821477,0.625899,False,low,high,high
3,4.0,314,0.571427,173,0.800505,0.495377,0.229078,0.800505,0.495377,False,high,high,high
4,5.0,419,0.449052,177,0.797731,0.482062,0.348679,0.797731,0.482062,False,high,high,high
5,6.0,335,0.459115,190,0.739779,0.466413,0.280664,0.739779,0.466413,False,high,high,high
6,7.0,327,0.565593,137,0.838918,0.565376,0.273325,0.838918,0.565376,False,high,high,high
7,8.0,348,0.550462,184,0.682690,0.300423,0.132228,0.682690,0.300423,False,mid,high,mid
8,9.0,362,0.432787,179,0.615036,0.184237,0.182249,0.615036,0.184237,False,high,mid,low
9,10.0,297,0.738340,150,0.784774,0.502307,0.046433,0.784774,0.502307,False,low,high,high


In [21]:
# --- Segment 4: Finalize classification, attach names, write output ---

name_lookup = crosswalk[["district_code", "district_name_nfhs4", "district_name_nfhs5"]].drop_duplicates()
district_intensity_out = district_intensity.merge(
    name_lookup, left_on="stable_district_id", right_on="district_code", how="left"
).drop(columns=["district_code"])

# Final coverage check before writing
n_missing_any_measure = district_intensity_out[
    ["delta_mobile_ownership", "nfhs5_mobile_level", "nfhs5_internet_level"]
].isna().any(axis=1).sum()
print(f"Districts missing at least one candidate measure: {n_missing_any_measure} / {len(district_intensity_out)}")

print("\nFinal column list:", district_intensity_out.columns.tolist())
print("\nFinal shape:", district_intensity_out.shape)

OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUT_DIR / "district_intensity_variables.parquet"
district_intensity_out.to_parquet(out_path, index=False)
print(f"\nWrote {out_path} ({district_intensity_out.shape})")

district_intensity_out.sort_values("stable_district_id").head(10)

Districts missing at least one candidate measure: 0 / 575

Final column list: ['stable_district_id', 'n_resp_nfhs4', 's930_mean', 'n_resp_nfhs5', 'v169a_mean', 's934_mean', 'delta_mobile_ownership', 'nfhs5_mobile_level', 'nfhs5_internet_level', 'low_n_either_round', 'delta_mobile_ownership_tercile', 'nfhs5_mobile_level_tercile', 'nfhs5_internet_level_tercile', 'district_name_nfhs4', 'district_name_nfhs5']

Final shape: (575, 15)

Wrote /Users/eknoorsandhu/DHS_India_Research/data/processed/district_intensity_variables.parquet ((575, 15))


,stable_district_id,n_resp_nfhs4,s930_mean,n_resp_nfhs5,v169a_mean,s934_mean,delta_mobile_ownership,nfhs5_mobile_level,nfhs5_internet_level,low_n_either_round,delta_mobile_ownership_tercile,nfhs5_mobile_level_tercile,nfhs5_internet_level_tercile,district_name_nfhs4,district_name_nfhs5
0,1.0,305,0.482354,182,0.709814,0.454159,0.227460,0.709814,0.454159,False,high,high,high,kupwara,kupwara
1,2.0,361,0.529261,189,0.679513,0.446047,0.150252,0.679513,0.446047,False,high,high,high,badgam,badgam
2,3.0,242,0.872892,177,0.821477,0.625899,-0.051416,0.821477,0.625899,False,low,high,high,leh,leh(ladakh)
3,4.0,314,0.571427,173,0.800505,0.495377,0.229078,0.800505,0.495377,False,high,high,high,kargil,kargil
4,5.0,419,0.449052,177,0.797731,0.482062,0.348679,0.797731,0.482062,False,high,high,high,punch,punch
5,6.0,335,0.459115,190,0.739779,0.466413,0.280664,0.739779,0.466413,False,high,high,high,rajouri,rajouri
6,7.0,327,0.565593,137,0.838918,0.565376,0.273325,0.838918,0.565376,False,high,high,high,kathua,kathua
7,8.0,348,0.550462,184,0.682690,0.300423,0.132228,0.682690,0.300423,False,mid,high,mid,baramula,baramula
8,9.0,362,0.432787,179,0.615036,0.184237,0.182249,0.615036,0.184237,False,high,mid,low,bandipore,bandipore
9,10.0,297,0.738340,150,0.784774,0.502307,0.046433,0.784774,0.502307,False,low,high,high,srinagar,srinagar


In [23]:
# --- Segment 5: Approximate standard error for delta_mobile_ownership ---

p4, n4 = district_intensity_out["s930_mean"], district_intensity_out["n_resp_nfhs4"]
p5, n5 = district_intensity_out["v169a_mean"], district_intensity_out["n_resp_nfhs5"]

# SE(delta) = sqrt(Var(p4) + Var(p5)), each Var(p) = p(1-p)/n -- standard
# binomial-proportion SE, combined in quadrature since the two rounds are
# independent samples
district_intensity_out["delta_mobile_ownership_se"] = np.sqrt(
    (p4 * (1 - p4) / n4) + (p5 * (1 - p5) / n5)
)

print("delta_mobile_ownership_se summary:")
print(district_intensity_out["delta_mobile_ownership_se"].describe())

# --- Rough gut-check: how many districts' deltas are noise-dominated? ---
# Not a rigorous test -- just a quick sense of scale
noisy_1se = (district_intensity_out["delta_mobile_ownership"].abs()
             < district_intensity_out["delta_mobile_ownership_se"]).sum()
noisy_2se = (district_intensity_out["delta_mobile_ownership"].abs()
             < 2 * district_intensity_out["delta_mobile_ownership_se"]).sum()
print(f"\nDistricts where |delta| < 1 SE: {noisy_1se} / {len(district_intensity_out)}")
print(f"Districts where |delta| < 2 SE (not distinguishable from zero at ~95%): {noisy_2se} / {len(district_intensity_out)}")

# --- Re-write the output file with the SE column added ---
district_intensity_out.to_parquet(out_path, index=False)
print(f"\nRe-wrote {out_path} with delta_mobile_ownership_se added ({district_intensity_out.shape})")

cols_to_show = ["stable_district_id", "delta_mobile_ownership", "delta_mobile_ownership_se"]
(
    district_intensity_out[cols_to_show]
    .sort_values("stable_district_id")
    .head(10)
)

delta_mobile_ownership_se summary:
count    575.000000
mean       0.051599
std        0.006806
min        0.030203
25%        0.048009
50%        0.051664
75%        0.055206
max        0.112021
Name: delta_mobile_ownership_se, dtype: float64

Districts where |delta| < 1 SE: 129 / 575
Districts where |delta| < 2 SE (not distinguishable from zero at ~95%): 275 / 575

Re-wrote /Users/eknoorsandhu/DHS_India_Research/data/processed/district_intensity_variables.parquet with delta_mobile_ownership_se added ((575, 16))


,stable_district_id,delta_mobile_ownership,delta_mobile_ownership_se
0,1.0,0.227460,0.044163
1,2.0,0.150252,0.042923
2,3.0,-0.051416,0.035875
3,4.0,0.229078,0.041268
4,5.0,0.348679,0.038757
5,6.0,0.280664,0.041886
6,7.0,0.273325,0.041686
7,8.0,0.132228,0.043455
8,9.0,0.182249,0.044731
9,10.0,0.046433,0.042149
